<a href="https://colab.research.google.com/github/Geethika1205/Generative_AI_2025/blob/main/lane_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files

uploaded = files.upload()


Saving yt1z.net - Lane Detection Test Video 01 (360p).mp4 to yt1z.net - Lane Detection Test Video 01 (360p).mp4


In [2]:
import os

video_path = list(uploaded.keys())[0]
print(f"Video uploaded: {video_path}")


Video uploaded: yt1z.net - Lane Detection Test Video 01 (360p).mp4


In [3]:
from IPython.display import HTML
from base64 import b64encode

video_path = "/content/yt1z.net - Lane Detection Test Video 01 (360p).mp4"  # Replace with your actual video filename

def display_video(video_path):
    with open(video_path, "rb") as video_file:
        video_data = video_file.read()
    video_base64 = b64encode(video_data).decode("utf-8")
    video_html = f"""
    <video width="640" height="480" controls>
        <source src="data:video/mp4;base64,{video_base64}" type="video/mp4">
    </video>
    """
    return HTML(video_html)

display_video(video_path)


In [11]:
import cv2
import numpy as np

def detect_lanes_with_hough(frame):
    """
    Detects lanes on the road using improved Hough Transform and ROI selection.
    :param frame: Input video frame
    :return: Processed frame with lanes detected
    """
    # Convert to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Apply Gaussian Blur to reduce noise
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    # Apply Canny edge detection
    edges = cv2.Canny(blurred, 50, 150)

    # Define new Region of Interest (ROI) mask
    mask = np.zeros_like(edges)
    height, width = edges.shape

    # Focus only on the road area (adjust based on road perspective)
    roi_vertices = np.array([
        [(width // 6, height), (width // 2.2, height // 1.4),
         (width // 1.8, height // 1.4), (5 * width // 6, height)]
    ], dtype=np.int32)

    cv2.fillPoly(mask, roi_vertices, 255)
    masked_edges = cv2.bitwise_and(edges, mask)

    # Apply Hough Transform with optimized parameters
    lines = cv2.HoughLinesP(masked_edges, 1, np.pi/180, threshold=50, minLineLength=100, maxLineGap=50)

    # Create a blank image for detected lines
    line_image = np.zeros_like(frame)

    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]

            # Calculate slope and filter lines to avoid horizontal or steeply angled ones
            if x2 - x1 != 0:
                slope = (y2 - y1) / (x2 - x1)
                if 0.4 < abs(slope) < 2:  # Allow only realistic lane angles
                    cv2.line(line_image, (x1, y1), (x2, y2), (0, 255, 0), 3)

    # Overlay the detected lines on the original frame
    result = cv2.addWeighted(frame, 0.8, line_image, 1, 0)

    return result

# Process video and save output
input_video = '/content/yt1z.net - Lane Detection Test Video 01 (360p).mp4'
output_video = 'lane_detection_result.mp4'

cap = cv2.VideoCapture(input_video)
frame_width = int(cap.get(3))
frame_height = int(cap.get(4))
fps = int(cap.get(cv2.CAP_PROP_FPS))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video, fourcc, fps, (frame_width, frame_height))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    processed_frame = detect_lanes_with_hough(frame)
    out.write(processed_frame)

cap.release()
out.release()
cv2.destroyAllWindows()

print(f'Processed video saved as {output_video}')


Processed video saved as lane_detection_result.mp4


In [12]:
from google.colab import files
files.download("/content/lane_detection_result.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>